In [1]:
import os
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


In [2]:
cols_keep = [
 'source_id',
 'ra',
 'dec',
 'parallax',
 'parallax_error',
 'astrometric_excess_noise',
 'astrometric_excess_noise_sig',
 'ipd_gof_harmonic_amplitude',
 'phot_bp_mean_flux_over_error',
 'phot_rp_mean_flux_over_error',
 #'phot_g_mean_flux',
 #'phot_g_mean_flux_error',
 'phot_g_mean_mag',
 'phot_bp_mean_mag',
 'phot_rp_mean_mag',
 'phot_bp_rp_excess_factor',
 'l',
 'b'
 ]

In [ ]:

def process_parquet_files(
    input_dir: str,
    output_dir: str,
    columns_to_save: list
):  
    c_tot, c_pos, c_pos_cl, c_neg, c_neg_cl = 0, 0, 0, 1e-5, 1e-5  # to avoid division by zero

    # Create output folder
    os.makedirs(output_dir, exist_ok=True)

    # Iterate through all parquet files in input_dir
    for file in os.listdir(input_dir):
        if not file.endswith(".parquet"):
            continue
        
        print(f'Processing file {file} ...')

        in_path = os.path.join(input_dir, file)
        df = pd.read_parquet(in_path)
        count_total = len(df)
        print(f'Total counts = {count_total}')
        c_tot += count_total

        # ---- Positive parallax selection ----
        df_pos = df[df["parallax"] > 0].copy()
        count_pos_total = len(df_pos)
        c_pos += count_pos_total

        # ipd condition
        
        # Already applied in ADQL
        #x = df_pos["parallax"] / df_pos["parallax_error"]
        #threshold = 10 ** (-5.12) * np.power(x, 2.61)
        #df_pos_sel = df_pos[df_pos["ipd_gof_harmonic_amplitude"] < threshold] 
        #df_pos_sel = df_pos[df_pos["parallax"] > 1.67]

        df_pos_sel = df_pos_sel[columns_to_save]
        count_pos_sel = len(df_pos_sel)
        c_pos_cl += count_pos_sel

        print(f'Selected {count_pos_sel} out of {count_pos_total}')

        df_sel = df_pos_sel

        # ---- Negative parallax selection ----
        df_neg = df[df["parallax"] < 0].copy()
        count_neg_total = len(df_neg)
        c_neg += count_neg_total

        if not df_neg.empty:
            # Already applied in ADQL

            #df_neg_sel = df_neg[
            #    (df_neg["astrometric_excess_noise"] < 1)
            #    & (df_neg["astrometric_excess_noise_sig"] > 2)
            #]
            #df_neg_sel = df_neg[df_neg["parallax"] > 1.67]

            if not df_neg_sel.empty:
                df_neg_sel = df_neg_sel[columns_to_save]
                count_neg_sel = len(df_neg_sel)
                c_neg_cl += count_neg_sel

                df_sel = pd.concat([df_sel, df_neg_sel], ignore_index=True)

                print(f'Selected {count_neg_sel} out of {count_neg_total}')
        
        out_path = os.path.join(output_dir, file)
        df_sel.to_parquet(out_path, index=False)

        print(f'Saved selection to {out_path}')

        print()
    
    print('All files processed.\n')
    print('Total star counts in preselection: ',c_tot)
    print(f'Total positive parallaxes: {c_pos}, ({round(c_pos/c_tot*100,2)}%)')
    print(f'Total negative parallaxes: {c_neg}, ({round(c_neg/c_tot*100,2)}%)')
    print(f'Selected positive parallaxes: {c_pos_cl}, ({round(c_pos_cl/c_pos*100,2)}%, {round(c_pos_cl/c_tot*100,2)}%)')
    print(f'Selected negative parallaxes: {c_neg_cl}, ({round(c_neg_cl/c_neg*100,2)}%, {round(c_neg_cl/c_tot*100,2)}%)')
    print(f'Total selected: {int(c_pos_cl + c_neg_cl)}, ({round((c_pos_cl + c_neg_cl)/c_tot*100,2)}%)')

In [4]:
process_parquet_files(
    input_dir="./data/600pc",
    output_dir="./data/600pc",
    columns_to_save=cols_keep
)

Processing file preselection_v2.parquet ...
Total counts = 1670101
Selected 1670101 out of 1670101
Saved selection to ./data/600pc/preselection_v2.parquet

All files processed.

Total star counts in preselection:  1670101
Total positive parallaxes: 1670101, (100.0%)
Total negative parallaxes: 1e-05, (0.0%)
Selected positive parallaxes: 1670101, (100.0%, 100.0%)
Selected negative parallaxes: 1e-05, (100.0%, 0.0%)
Total selected: 1670101.00001, (100.0%)
